# The PFT and IPFT

`pypft.forward_pft` and `pypft.inverse_pft` run the whole chain of [04_domains](04_domains.ipynb) in one call, on plain arrays:

$$f(r, \theta) \xrightarrow{\text{DFT}_\theta} f_n(r) \xrightarrow{H_n} F_n(\rho) \xrightarrow{\text{IDFT}_\psi} F(\rho, \psi)$$

This notebook checks both directions against an exact continuous transform pair, reproducing the accuracy Yao & Baddour report for it [[1]](#ref-1) (Appendix A-5/A-6), looks at where the error lives, for a centered and an off-center function, and shows what an inadequate grid does to the transform, and what it does not do to a round trip.

In [ ]:
# Draw every figure below the cell that makes it
%matplotlib inline

# The warnings module to record the grid checks, NumPy, matplotlib, and PyPFT itself
import warnings

import matplotlib.pyplot as plt
import numpy as np

import pypft

## The forward transform

`forward_pft(f, grid)` takes an array of shape `(n_radial, n_angular)` on PyPFT's `(radial, angular)` layout. `grid.r` has the opposite layout, one row of radii per spoke, so `grid.r.T` gives the radius of every sample of `f`.

The grid is Yao & Baddour's worked example: $N_2 = 15$ angular samples, $N_1 = 383$ (so `n_radial` $= N_1 - 1 = 382$ radial samples), and the space limit $R = 40$ [[1]](#ref-1) (Appendix A-5). The test function is the Gaussian $f(r) = e^{-r^2}$, circularly symmetric and negligible long before $r = R$.

In [ ]:
# Yao & Baddour's worked example
grid = pypft.PolarGrid(n_radial=382, n_angular=15, R=40.0)

# The Gaussian, sampled on the (radial, angular) layout
f = np.exp(-(grid.r.T**2))

# The forward transform, and both signals wrapped for plotting
F = pypft.forward_pft(f=f, grid=grid)
f_signal = pypft.PolarSpatialSignal(values=f, grid=grid)
F_signal = pypft.PolarFrequencySignal(values=F, grid=grid)
print(f"f: {f.shape}, F: {F.shape}")

Both signals, as magnitude and phase on the polar index table, and as an image in the plane. `pypft.render_cartesian` interpolates the polar samples onto a pixel grid for display only; its output is never fed back into a transform. The Gaussian is concentrated near the center of a large disk, so the magnitudes are drawn with `plot_signal`'s gamma-enhanced color scale, and the Cartesian images below zoom in on the center. The phase of $F$ is only meaningful where its magnitude is not negligible.

In [ ]:
# Top row: f; bottom row: F; magnitude and phase on the index table, then the plane
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 8))
for signal, row, half_width in ((f_signal, axes[0], 3.0), (F_signal, axes[1], 12.0)):
    pypft.plot_signal(signal=signal, ax=(row[0], row[1]))
    pypft.render_cartesian(signal=signal, height=400, width=400, ax=row[2])
    # Zoom in on the part of the disk where the signal lives
    row[2].set_xlim(left=-half_width, right=half_width)
    row[2].set_ylim(bottom=-half_width, top=half_width)
fig.tight_layout()
plt.show()

## Comparing with the exact transform

A circularly symmetric Gaussian has a closed-form 2-D Fourier transform: $f(r) = e^{-r^2}$ transforms into $F(\rho) = \pi e^{-\rho^2/4}$ [[1]](#ref-1) (Appendix A-5). The error at each sample is measured in dB relative to the largest value of $F$, as in the paper:

$$E = 20 \log_{10} \frac{\lvert F_\text{exact} - F \rvert}{\max \lvert F \rvert}.$$

The paper summarizes it by its average $E_\text{avg}$ and its maximum $E_\text{max}$ over all samples. Neither says how far the transform as a whole is from the exact one. $E_\text{avg}$ averages a logarithm over every sample, so it is dominated by the many samples where both values are essentially zero, as the last section of [02_sampling_grids](02_sampling_grids.ipynb) shows. The relative L2 error does say it, so it is reported alongside:

$$\frac{\lVert F - F_\text{exact} \rVert}{\lVert F_\text{exact} \rVert}.$$

In [ ]:
# The exact transform at every frequency sample
F_exact = np.pi * np.exp(-(grid.rho.T**2) / 4)

# The error of every sample, in dB relative to the peak of F
err_db = 20 * np.log10(np.abs(F_exact - F) / np.max(a=np.abs(F)))
print(f"E_avg: {err_db.mean():.2f} dB")
print(f"E_max: {err_db.max():.2f} dB")

# The relative L2 error of the whole transform
relative_l2 = np.linalg.norm(x=F - F_exact) / np.linalg.norm(x=F_exact)
print(f"relative L2 error: {relative_l2:.3f}")

These are Yao & Baddour's published figures for this grid [[1]](#ref-1) (Appendix A-5). The maximum is far worse than the average, and the figures show why. On the index table (left) the error is worst in the first rows, the smallest radial frequencies. In the plane (middle) those samples ring the central gap of the frequency grid, the hole around $\rho = 0$ that [02_sampling_grids](02_sampling_grids.ipynb) explains. Plotted against $\rho$ (right), the error is largest at the smallest radial frequencies, near $-9$ dB, and falls by more than 100 dB toward the band limit; the histogram underneath shows how few samples are anywhere near the maximum. Reproducing the paper's figures is therefore about $E_\text{avg}$, never $E_\text{max}$. How close the transform is to the continuous one is what the relative L2 error measures, and at about $0.24$ it is far from the $-64$ dB the average suggests. Most of that error sits at the smallest radial frequencies, where $F$ is largest; it comes from the spurious angular harmonics that on-grid samples of this circularly symmetric function carry, shown in [02_sampling_grids](02_sampling_grids.ipynb).

In [ ]:
# The frequency-plane position of every sample (y down, as in render_cartesian)
rho = grid.rho.T
psi = grid.psi[np.newaxis, :]
x, y = rho * np.cos(psi), -rho * np.sin(psi)

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4.5))

# Left: the error on the polar index table
shown = axes[0].imshow(X=err_db, aspect="auto", origin="lower", cmap="magma")
axes[0].set_xlabel(xlabel="angular index")
axes[0].set_ylabel(ylabel="radial index")
axes[0].set_title(label="error on the index table (dB)")
fig.colorbar(mappable=shown, ax=axes[0])

# Middle: the error at every sample's position in the frequency plane, near the center
shown = axes[1].scatter(x=x, y=y, c=err_db, s=12, cmap="magma")
axes[1].set_xlim(left=-1.5, right=1.5)
axes[1].set_ylim(bottom=-1.5, top=1.5)
axes[1].set_aspect(aspect="equal")
axes[1].set_xlabel(xlabel="x")
axes[1].set_ylabel(ylabel="y")
axes[1].set_title(label="error near the center of the plane (dB)")
fig.colorbar(mappable=shown, ax=axes[1])

# Right: the error against radial frequency, with the average marked
axes[2].scatter(x=rho, y=err_db, s=4)
axes[2].axhline(y=err_db.mean(), color="black", linestyle="--", label="$E_{avg}$")
axes[2].set_xlabel(xlabel=r"$\rho$")
axes[2].set_ylabel(ylabel="dB")
axes[2].set_title(label="error against radial frequency")
axes[2].legend()
fig.tight_layout()
plt.show()

# How the errors are distributed, with the average and the maximum marked
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(x=err_db.ravel(), bins=60)
ax.axvline(x=err_db.mean(), color="black", linestyle="--", label="$E_{avg}$")
ax.axvline(x=err_db.max(), color="red", linestyle=":", label="$E_{max}$")
ax.set_xlabel(xlabel="dB")
ax.set_ylabel(ylabel="samples")
ax.legend()
plt.show()

## The inverse transform

`inverse_pft(F, grid)` runs the three steps in the opposite direction,

$$F(\rho, \psi) \xrightarrow{\text{DFT}_\psi} F_n(\rho) \xrightarrow{H_n^{-1}} f_n(r) \xrightarrow{\text{IDFT}_\theta} f(r, \theta).$$

The same pair checks it the other way round: feed in the exact frequency-domain Gaussian and compare the result with the exact space-domain one [[1]](#ref-1) (Appendix A-6).

In [ ]:
# The inverse transform of the exact frequency-domain Gaussian
f_reconstructed = pypft.inverse_pft(F=F_exact, grid=grid)

# The error of every sample, in dB relative to the peak of the result
inverse_err_db = 20 * np.log10(
    np.abs(f - f_reconstructed) / np.max(a=np.abs(f_reconstructed))
)
print(f"E_avg: {inverse_err_db.mean():.2f} dB")
print(f"E_max: {inverse_err_db.max():.2f} dB")

# The relative L2 error of the whole inverse transform
inverse_relative_l2 = np.linalg.norm(x=f_reconstructed - f) / np.linalg.norm(x=f)
print(f"relative L2 error: {inverse_relative_l2:.3f}")

## Inadequate grids

Two warnings flag a grid that is too coarse for the transform:

- `pypft.check_adequacy(grid)` warns (`AdequacyWarning`) when `n_radial` is too small for `n_angular`, from a fit of the measured forward error of this same Gaussian.
- `pypft.check_nyquist_adequacy(grid, band_limit)` warns (`NyquistWarning`) when the grid cannot hold a function whose transform extends up to `band_limit` [[1]](#ref-1) (Eq. 21). The transform $\pi e^{-\rho^2/4}$ has dropped below $10^{-15}$ by $\rho = 12$, so the band limit used here is $12$.

The cell builds three grids with $R = 40$: the worked example; one with four times as many angular samples but the same radial samples, which fails the first check; and one with only 96 radial samples, which fails both. Both checks only ever warn, so the cell records their warnings with `warnings.catch_warnings`, prints them, and goes on to transform the Gaussian on every grid.

In [ ]:
# The worked example, more harmonics than its radial samples support, and too few radii
grids = {
    "adequate": pypft.PolarGrid(n_radial=382, n_angular=15, R=40.0),
    "too many harmonics": pypft.PolarGrid(n_radial=382, n_angular=63, R=40.0),
    "too few radii": pypft.PolarGrid(n_radial=96, n_angular=15, R=40.0),
}

results = {}
for name, candidate in grids.items():
    # Record every warning the two checks emit, whatever filters are active
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter(action="always")
        pypft.check_adequacy(grid=candidate)
        pypft.check_nyquist_adequacy(grid=candidate, band_limit=12.0)
    print(f"{name} ({candidate.n_radial} x {candidate.n_angular}):")
    for warning in caught:
        print(f"    {warning.category.__name__}: {warning.message}")

    # The forward error against the exact transform, and the round-trip error
    f_candidate = np.exp(-(candidate.r.T**2))
    F_candidate = pypft.forward_pft(f=f_candidate, grid=candidate)
    F_candidate_exact = np.pi * np.exp(-(candidate.rho.T**2) / 4)
    forward_db = 20 * np.log10(
        np.abs(F_candidate_exact - F_candidate) / np.max(a=np.abs(F_candidate))
    )
    round_trip = pypft.inverse_pft(F=F_candidate, grid=candidate)
    round_trip_error = np.abs(round_trip - f_candidate)
    results[name] = (forward_db, round_trip_error)
    forward_l2 = np.linalg.norm(x=F_candidate - F_candidate_exact) / np.linalg.norm(
        x=F_candidate_exact
    )
    print(f"    forward E_avg: {forward_db.mean():.2f} dB, relative L2: {forward_l2:.3f}")
    print(f"    max round-trip error: {round_trip_error.max():.1e}")

The forward error grows by tens of dB on both inadequate grids, and the relative L2 error from about $0.24$ to about $0.65$, yet the round trip `inverse_pft(forward_pft(f))` returns `f` to rounding error on all three. The figure shows both errors on every grid: forward on top, round trip below, on the polar index table.

The round trip cannot see the problem. The DHT matrix is its own inverse and the forward and inverse scale factors cancel, so the round trip undoes the forward transform exactly whether or not that transform approximates the continuous one [[1]](#ref-1) (Appendix A-5/A-6). A passing round trip only shows that the inverse undoes the forward; accuracy is shown by comparing with an exact transform, as above.

In [ ]:
# One column per grid: forward error on top, round-trip error below
fig, axes = plt.subplots(nrows=2, ncols=len(results), figsize=(15, 8))
for column, (name, (forward_db, round_trip_error)) in enumerate(results.items()):
    # Top: the forward error in dB, on one shared color scale
    shown = axes[0, column].imshow(
        X=forward_db, aspect="auto", origin="lower", cmap="magma", vmin=-160, vmax=20
    )
    axes[0, column].set_title(label=f"{name}: forward error (dB)")
    fig.colorbar(mappable=shown, ax=axes[0, column])

    # Bottom: the round-trip error, on a log scale
    shown = axes[1, column].imshow(
        X=np.log10(round_trip_error + 1e-20),
        aspect="auto",
        origin="lower",
        vmin=-20,
        vmax=-10,
    )
    axes[1, column].set_title(label=f"{name}: round-trip error (log10)")
    fig.colorbar(mappable=shown, ax=axes[1, column])
for ax in axes.ravel():
    ax.set_xlabel(xlabel="angular index")
    ax.set_ylabel(ylabel="radial index")
fig.tight_layout()
plt.show()

## An off-center function

The Gaussian above is circularly symmetric, so every harmonic but $0$ of its exact transform vanishes. A Gaussian centered away from the origin, $f = e^{-\lvert \mathbf{x} - \mathbf{x}_0 \rvert^2}$ with $\mathbf{x}_0 = (2, -1)$, has every harmonic, and its exact transform $\pi e^{-\rho^2/4} e^{-i \boldsymbol{\rho} \cdot \mathbf{x}_0}$ is complex. On the same worked-example grid, $E_\text{avg}$ looks even better than for the centered Gaussian, while the relative L2 error stays at about $0.24$.

The figure shows where that error sits. Left: the size of the error at every sample against $\rho$, next to the size of the exact transform. Middle: each spoke's share of the total squared error. Right: the same errors in dB, the quantity $E_\text{avg}$ averages. The error lives at small $\rho$, where the transform itself is large, and it is present on every spoke, most of all on the spokes with the largest $|q|$, not only in the central gap. In dB those samples are a small minority: the thousands of samples at large $\rho$, where both values are essentially zero, sit between about $-80$ and $-160$ dB and pull the average down.

The cause is the one [02_sampling_grids](02_sampling_grids.ipynb) shows. The transform identifies each spoke's angular sample index with a harmonic order, so on both the space and the frequency side, samples of one radial index sit at different radii on different spokes; the error is small only where the function varies slowly across those offsets. More radial samples do not remove it.

In [ ]:
# A Gaussian centered at (2, -1), and its exact transform
x0, y0 = 2.0, -1.0
r, theta = grid.r.T, grid.theta[np.newaxis, :]
f_off = np.exp(-((r * np.cos(theta) - x0) ** 2 + (r * np.sin(theta) - y0) ** 2))
phase = rho * (np.cos(psi) * x0 + np.sin(psi) * y0)
F_off_exact = np.pi * np.exp(-(rho**2) / 4) * np.exp(-1j * phase)

# The forward transform, its dB error and its relative L2 error
F_off = pypft.forward_pft(f=f_off + 0j, grid=grid)
error = np.abs(F_off - F_off_exact)
off_db = 20 * np.log10(error / np.max(a=np.abs(F_off)))
off_l2 = np.linalg.norm(x=F_off - F_off_exact) / np.linalg.norm(x=F_off_exact)
print(f"E_avg: {off_db.mean():.2f} dB, relative L2 error: {off_l2:.3f}")

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4.5))

# Left: the size of the error and of the exact transform against radial frequency
axes[0].scatter(x=rho, y=np.abs(F_off_exact), s=4, color="gray", label=r"$|F_{exact}|$")
axes[0].scatter(x=rho, y=error, s=4, color="tab:red", label=r"$|F - F_{exact}|$")
axes[0].set_xlim(left=0.0, right=8.0)
axes[0].set_xlabel(xlabel=r"$\rho$")
axes[0].set_ylabel(ylabel="magnitude")
axes[0].set_title(label="error and transform against radial frequency")
axes[0].legend()

# Middle: each spoke's share of the total squared error
share = np.sum(a=error**2, axis=0) / np.sum(a=error**2)
axes[1].bar(x=grid.harmonics, height=share)
axes[1].set_xlabel(xlabel="angular index $q$")
axes[1].set_ylabel(ylabel="share of the squared error")
axes[1].set_title(label="squared error by spoke")

# Right: the dB error of every sample, with its average marked
axes[2].scatter(x=rho, y=off_db, s=4)
axes[2].axhline(y=off_db.mean(), color="black", linestyle="--", label="$E_{avg}$")
axes[2].set_xlabel(xlabel=r"$\rho$")
axes[2].set_ylabel(ylabel="dB")
axes[2].set_title(label="error in dB against radial frequency")
axes[2].legend()
fig.tight_layout()
plt.show()

## Where to go next

[06_batches](06_batches.ipynb) transforms a stack of signals in one call, with a trailing batch axis.

## References

1. <a id="ref-1"></a> X. Yao and N. Baddour, "Discrete Two-Dimensional Fourier Transform in Polar Coordinates Part II: Numerical Computation and Approximation of the Continuous Transform," *PeerJ Computer Science*, 6:e257, 2020. [doi:10.7717/peerj-cs.257](https://doi.org/10.7717/peerj-cs.257)